# Sleep Health & Lifestyle Analysis

**Project context:** completed as a DataCamp guided project. This notebook presents a concise, reproducible version of the analysis in my own words.

The supplied sample contains 374 anonymized records and 13 fields. The analysis compares average sleep duration and quality by occupation, then calculates the share of each BMI category recorded with insomnia. These are descriptive sample patterns, not clinical or causal conclusions.

## Load and inspect the data

Keep the literal `None` disorder value so it remains a category meaning no recorded disorder, rather than being interpreted as a missing value.

In [1]:
import pandas as pd

df = pd.read_csv("sleep_health_data.csv", keep_default_na=False)
print(f"Rows: {len(df)} | Columns: {df.shape[1]}")
print("\nSleep disorder values:\n", df["Sleep Disorder"].value_counts().to_string())

Rows: 374 | Columns: 13

Sleep disorder values:
None           219
Sleep Apnea     78
Insomnia        77


## Compare sleep averages by occupation

Calculate average sleep duration and quality for each occupation, while keeping each group size visible.

In [2]:
occupation_summary = (
    df.groupby("Occupation", as_index=False)
      .agg(people=("Person ID", "count"),
           avg_sleep_hours=("Sleep Duration", "mean"),
           avg_sleep_quality=("Quality of Sleep", "mean"))
)

lowest_duration = occupation_summary.loc[occupation_summary["avg_sleep_hours"].idxmin()]
lowest_quality = occupation_summary.loc[occupation_summary["avg_sleep_quality"].idxmin()]
print(f"Lowest average sleep duration: {lowest_duration['Occupation']} — {lowest_duration['avg_sleep_hours']:.1f} hours (n={lowest_duration['people']})")
print(f"Lowest average sleep quality: {lowest_quality['Occupation']} — {lowest_quality['avg_sleep_quality']:.1f} / 10 (n={lowest_quality['people']})")

Lowest average sleep duration: Sales Representative — 5.9 hours (n=2)
Lowest average sleep quality: Sales Representative — 4.0 / 10 (n=2)


## Calculate insomnia share by BMI category

Use the total number of records in each BMI category as the denominator, then retain both case counts and percentages.

In [3]:
bmi_summary = (
    df.assign(is_insomnia=df["Sleep Disorder"].eq("Insomnia"))
      .groupby("BMI Category", as_index=False)
      .agg(people=("Person ID", "count"),
           insomnia_cases=("is_insomnia", "sum"))
)
bmi_summary["insomnia_share_pct"] = (bmi_summary["insomnia_cases"] / bmi_summary["people"] * 100).round(1)
print(bmi_summary.to_string(index=False))

BMI Category  people  insomnia_cases  insomnia_share_pct
      Normal     216               9                 4.2
  Overweight     148              64                43.2
       Obese      10               4                40.0


## Interpretation and limitations

Sales Representatives have the lowest average sleep duration (5.9 hours) and quality (4/10), but the group contains only two records. The Overweight group has the highest observed insomnia share (43.2%, 64 of 148); the Obese group is close at 40.0% (4 of 10).

These results describe this dataset only. Small occupation and BMI groups can produce unstable percentages, and the dataset cannot show that BMI or occupation causes sleep outcomes. Treat this as exploratory practice, not medical evidence.